# Notebook 8: Larger target models

**Reviewer request (reviewer 4):** *test larger models.* The paper's tree models have 100 trees. Here the loss is measured again for **larger** versions, and the meta-model is tested on them:

| Model | Standard (paper) | Larger (this notebook) |
|---|---|---|
| Random forest | 100 trees | 500 trees |
| XGBoost | 100 trees, depth 6 | 500 trees, depth 8, learning rate 0.05, subsample 0.8 |
| LightGBM | 100 trees, 31 leaves | 500 trees, 63 leaves, learning rate 0.05, subsample 0.8 |

**What it does**

1. For seeds 0-2 it trains the larger models on the clean data and on damaged copies of the training data (Round-1 datasets: the clean copy, label noise, missing values, duplicates, imbalance and the two combined settings; Round-2 datasets: the clean copy and label noise at 4 levels, which is all the planned test needs).
2. It trains the **standard-size** models itself on the same damaged data (no earlier files are needed), and re-uses the meta-features from Notebook 7 files (`n7_*`) if they are found; otherwise it computes them.
3. It runs the **same planned test** as the paper (train the meta-model on Round 1, test on label noise in Round 2), once for the standard models and once for the larger models, on the same seeds.
4. It also tests whether a meta-model trained on the **standard-size** models can predict the loss of the **larger** models.
5. It compares the training time of the larger models with the time to compute the meta-features.

**Running time:** several hours on Colab (it trains 6 models per setting). Everything is saved per dataset and seed, so you can stop and run the cell again. **First** set `FAST_TEST = True` (a few minutes) to check that everything works, then set it back to `False`.

In [ ]:
!pip install -q openml lightgbm cleanlab

## Setup: where files are saved

`N7_DIR` is the folder with the Notebook 7 files (`n7_*_seed*.csv`). Results of this notebook go to a new folder.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/data_quality_paper_v8'
    N7_DIR = '/content/drive/MyDrive/data_quality_paper_v7'
except ImportError:
    OUTPUT_DIR = 'results_v8'
    N7_DIR = 'results_v7'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Saving results to:', OUTPUT_DIR)
print('Notebook 7 files found:', len([f for f in os.listdir(N7_DIR) if f.startswith('n7_')]) if os.path.isdir(N7_DIR) else 0)

In [ ]:
import time, glob, warnings
import numpy as np
import pandas as pd
import openml

warnings.filterwarnings('ignore')

FAST_TEST = False                 # True = only 4 small datasets, to test that everything works
SEEDS = [0, 1, 2]
STD_MODELS = ['rf', 'xgb', 'lgbm']
BIG_MODELS = ['rf_big', 'xgb_big', 'lgbm_big']
PAIR = dict(zip(BIG_MODELS, STD_MODELS))
METRICS = ['acc', 'bacc', 'f1']
METRIC_NAMES = {'acc': 'accuracy', 'bacc': 'balanced accuracy', 'f1': 'macro-F1'}
FILE_PREFIX = 'n8'

ORIGINAL_IDS = [3, 15, 29, 31, 37, 38, 44, 50, 1049, 1067, 1480,
                11, 23, 46, 54, 307, 12, 14, 16, 22, 28]
NEW_IDS = [18, 182, 188, 458, 469, 1050, 1063, 1068, 1462, 1475, 1487, 1489, 1494, 1497,
           1510, 1464, 4538, 6332, 23381, 40499, 40701, 40966, 40975, 40982, 40983, 40984, 40994]
DATASET_IDS = ORIGINAL_IDS + NEW_IDS
if FAST_TEST:
    ORIGINAL_IDS = ORIGINAL_IDS[:2]
    NEW_IDS = [1063, 1464]
    DATASET_IDS = ORIGINAL_IDS + NEW_IDS


def load_dataset(did):
    ds = openml.datasets.get_dataset(did)
    X, y, _, _ = ds.get_data(target=ds.default_target_attribute)
    for col in X.columns:
        if not pd.api.types.is_numeric_dtype(X[col]):
            codes = X[col].astype('category').cat.codes.astype(float)
            X[col] = codes.where(codes >= 0, np.nan)
    X = X.astype(float).reset_index(drop=True)
    y = pd.factorize(y)[0]
    return X, y, ds.name

## Part A: data problems and models

The defects, levels and settings are exactly those of Notebooks 6 and 7, so the damaged training sets are identical.

In [ ]:
def inject_missing(X, level, seed=0):
    rng = np.random.default_rng(seed)
    return X.mask(rng.random(X.shape) < level)


def inject_label_noise(y, level, seed=0):
    rng = np.random.default_rng(seed)
    y = y.copy()
    classes = np.unique(y)
    idx = rng.choice(len(y), int(level * len(y)), replace=False)
    for i in idx:
        y[i] = rng.choice(classes[classes != y[i]])
    return y


def inject_class_noise(y, level, seed=0):
    # class-dependent noise: a label moves to the next class (cyclic)
    rng = np.random.default_rng(seed + 12345)
    y = y.copy()
    classes = np.unique(y)
    pos = {c: i for i, c in enumerate(classes)}
    idx = rng.choice(len(y), int(level * len(y)), replace=False)
    for i in idx:
        y[i] = classes[(pos[y[i]] + 1) % len(classes)]
    return y


def inject_duplicates(X, y, level, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), int(level * len(X)), replace=True)
    return pd.concat([X, X.iloc[idx]], ignore_index=True), np.concatenate([y, y[idx]])


def inject_imbalance(X, y, level, seed=0):
    rng = np.random.default_rng(seed)
    classes, counts = np.unique(y, return_counts=True)
    biggest = classes[np.argmax(counts)]
    keep = []
    for c in classes:
        idx = np.where(y == c)[0]
        if c != biggest:
            idx = rng.choice(idx, max(2, int(round(len(idx) * (1 - level)))), replace=False)
        keep.extend(idx)
    keep = np.sort(np.array(keep))
    return X.iloc[keep].reset_index(drop=True), y[keep]


def apply_defect(X_tr, y_tr, defect, level, seed=0):
    if defect == 'missing':
        return inject_missing(X_tr, level, seed), y_tr
    if defect == 'label_noise':
        return X_tr, inject_label_noise(y_tr, level, seed)
    if defect == 'class_noise':
        return X_tr, inject_class_noise(y_tr, level, seed)
    if defect == 'duplicates':
        return inject_duplicates(X_tr, y_tr, level, seed)
    if defect == 'imbalance':
        return inject_imbalance(X_tr, y_tr, level, seed)
    raise ValueError('unknown defect: ' + defect)


DEFECTS = ['missing', 'label_noise', 'class_noise', 'duplicates', 'imbalance']
DEFECT_ORDER = ['label_noise', 'class_noise', 'imbalance', 'duplicates', 'missing']
COMBINED = ['missing', 'label_noise', 'duplicates', 'imbalance']   # as in the original paper
LEVELS = [0.05, 0.1, 0.2, 0.4]
COMBO_LEVELS = [0.1, 0.2]


def apply_defects(X, y, setting, seed=0):
    for d in DEFECT_ORDER:
        if d in setting:
            X, y = apply_defect(X, y, d, setting[d], seed)
    return X, y


def build_settings():
    settings = [{}]
    for d in DEFECTS:
        for lv in LEVELS:
            settings.append({d: lv})
    for lv in COMBO_LEVELS:
        settings.append({d: lv for d in COMBINED})
    return settings


SETTINGS = build_settings()
print('Number of settings per dataset:', len(SETTINGS))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


def make_model(name):
    imputer = SimpleImputer(strategy='median')
    if name == 'logreg':
        return make_pipeline(imputer, StandardScaler(), LogisticRegression(max_iter=1000))
    if name == 'rf':
        return make_pipeline(imputer, RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=0))
    if name == 'xgb':
        return make_pipeline(imputer, XGBClassifier(n_estimators=100, n_jobs=-1, random_state=0, verbosity=0))
    if name == 'lgbm':
        return make_pipeline(imputer, LGBMClassifier(n_estimators=100, n_jobs=-1, random_state=0, verbose=-1))
    if name == 'mlp':
        return make_pipeline(imputer, StandardScaler(), MLPClassifier(
            hidden_layer_sizes=(64,), max_iter=200, early_stopping=True, random_state=0))
    raise ValueError('unknown model: ' + name)


def fit_predict(name, X_fit, y_fit, X_eval):
    # trains a model and predicts; the neural network falls back to a version without
    # early stopping if the internal validation split fails (tiny classes)
    try:
        return make_model(name).fit(X_fit, y_fit).predict(X_eval)
    except Exception:
        if name != 'mlp':
            raise
        model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), MLPClassifier(
            hidden_layer_sizes=(64,), max_iter=200, early_stopping=False, random_state=0))
        return model.fit(X_fit, y_fit).predict(X_eval)


def scores(y_true, y_pred):
    return {'acc': accuracy_score(y_true, y_pred),
            'bacc': balanced_accuracy_score(y_true, y_pred),
            'f1': f1_score(y_true, y_pred, average='macro')}

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


def make_big_model(name):
    imputer = SimpleImputer(strategy='median')
    if name == 'rf_big':
        return make_pipeline(imputer, RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=0))
    if name == 'xgb_big':
        return make_pipeline(imputer, XGBClassifier(n_estimators=500, max_depth=8, learning_rate=0.05, subsample=0.8,
                                                    colsample_bytree=0.8, n_jobs=-1, random_state=0, verbosity=0))
    if name == 'lgbm_big':
        return make_pipeline(imputer, LGBMClassifier(n_estimators=500, num_leaves=63, learning_rate=0.05, subsample=0.8,
                                                     subsample_freq=1, colsample_bytree=0.8, n_jobs=-1, random_state=0, verbose=-1))
    raise ValueError('unknown model: ' + name)


def fit_predict_big(name, X_fit, y_fit, X_eval):
    # larger models are trained here; standard-size models use the paper's settings
    if name in STD_MODELS:
        return make_model(name).fit(X_fit, y_fit).predict(X_eval)
    return make_big_model(name).fit(X_fit, y_fit).predict(X_eval)


ALL_MODELS = BIG_MODELS + STD_MODELS


def keep_setting(sid, round1):
    # Round 1: all settings except class-dependent noise. Round 2: clean + label noise only.
    s = SETTINGS[sid]
    if not s:
        return True
    if round1:
        return 'class_noise' not in s
    return list(s.keys()) == ['label_noise']


print('Settings used in Round 1:', sum(keep_setting(i, True) for i in range(len(SETTINGS))),
      '| in Round 2:', sum(keep_setting(i, False) for i in range(len(SETTINGS))))

## Part B: meta-features (re-used from Notebook 7 when possible)

In [ ]:
from scipy.stats import entropy
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

META_COLS = ['n_rows', 'n_features', 'n_classes', 'rows_per_feature',
             'missing_rate', 'duplicate_rate', 'imbalance_ratio', 'outlier_rate',
             'mean_skew', 'mean_kurtosis', 'mean_abs_corr',
             'class_entropy', 'mean_mutual_info',
             'lm_stump', 'lm_tree3', 'lm_nb']


def compute_meta_features(X, y, seed=0):
    rng = np.random.default_rng(seed)
    n, p = X.shape
    classes, counts = np.unique(y, return_counts=True)
    prop = counts / counts.sum()
    mf = {'n_rows': n, 'n_features': p, 'n_classes': len(classes), 'rows_per_feature': n / p,
          'missing_rate': float(X.isna().to_numpy().mean()),
          'duplicate_rate': float(X.assign(_y=y).duplicated().mean()),
          'imbalance_ratio': float(counts.min() / counts.max()),
          'class_entropy': float(entropy(prop) / np.log(len(classes))) if len(classes) > 1 else 0.0}
    A = X.to_numpy(dtype=float)
    mu = np.nanmean(A, axis=0)
    sd = np.nanstd(A, axis=0)
    sd[sd == 0] = np.nan
    z = np.abs((A - mu) / sd)
    valid = ~np.isnan(z)
    mf['outlier_rate'] = float((z[valid] > 3).mean()) if valid.any() else 0.0
    mf['mean_skew'] = float(X.skew().mean())
    mf['mean_kurtosis'] = float(X.kurt().mean())
    if p > 1:
        C = X.corr().abs().to_numpy()
        mf['mean_abs_corr'] = float(np.nanmean(C[np.triu_indices(p, k=1)]))
    else:
        mf['mean_abs_corr'] = 0.0
    m = min(n, 1000)
    idx = rng.choice(n, m, replace=False)
    t_lm = time.perf_counter()
    Xs = X.iloc[idx]
    ys = y[idx]
    Xi = Xs.fillna(Xs.median()).fillna(0.0).to_numpy()
    cols = rng.choice(p, min(p, 20), replace=False)
    try:
        mf['mean_mutual_info'] = float(mutual_info_classif(Xi[:, cols], ys, random_state=seed).mean())
    except Exception:
        mf['mean_mutual_info'] = np.nan

    def landmark(model):
        try:
            return float(cross_val_score(model, Xi, ys, cv=3).mean())
        except Exception:
            return np.nan

    mf['lm_stump'] = landmark(DecisionTreeClassifier(max_depth=1, random_state=0))
    mf['lm_tree3'] = landmark(DecisionTreeClassifier(max_depth=3, random_state=0))
    mf['lm_nb'] = landmark(GaussianNB())
    mf['t_landmark'] = time.perf_counter() - t_lm
    return mf

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, KFold
from cleanlab.filter import find_label_issues
from cleanlab.count import compute_confident_joint, estimate_joint

CL_COLS = ['cl_issue_rate', 'cl_noise_rate', 'cl_self_conf', 'cl_disagree']


def cl_pred_probs(Xi, ys, K, seed):
    # out-of-sample class probabilities from a small random forest (3 folds)
    n = len(ys)
    probs = np.zeros((n, K))
    counts = np.bincount(ys, minlength=K)
    folds = StratifiedKFold(3, shuffle=True, random_state=seed) if counts.min() >= 3 else KFold(3, shuffle=True, random_state=seed)
    for tr, te in folds.split(Xi, ys):
        clf = RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=seed).fit(Xi[tr], ys[tr])
        probs[np.ix_(te, clf.classes_)] = clf.predict_proba(Xi[te])
    return probs


def compute_cl_features(X, y, seed=0):
    out = {c: np.nan for c in CL_COLS}
    rng = np.random.default_rng(seed + 99)
    n = len(y)
    m = min(n, 1000)
    idx = rng.choice(n, m, replace=False)
    Xs = X.iloc[idx]
    Xi = Xs.fillna(Xs.median()).fillna(0.0).to_numpy()
    classes, ys = np.unique(np.asarray(y)[idx], return_inverse=True)
    K = len(classes)
    if K < 2:
        return out
    try:
        P = cl_pred_probs(Xi, ys, K, seed)
        out['cl_self_conf'] = float(P[np.arange(m), ys].mean())
        out['cl_disagree'] = float((P.argmax(axis=1) != ys).mean())
        P = P / np.clip(P.sum(axis=1, keepdims=True), 1e-12, None)
        out['cl_issue_rate'] = float(np.mean(find_label_issues(ys, P)))
        cj = compute_confident_joint(ys, P)
        out['cl_noise_rate'] = float(1.0 - np.trace(estimate_joint(ys, P, confident_joint=cj)))
    except Exception:
        pass
    return out


def all_meta_features(X, y, seed=0):
    t0 = time.perf_counter()
    mf = compute_meta_features(X, y, seed)
    mf['t_meta'] = time.perf_counter() - t0
    t1 = time.perf_counter()
    mf.update(compute_cl_features(X, y, seed))
    mf['t_cl'] = time.perf_counter() - t1
    return mf

## Part A (continued): run the larger models

One file is saved per dataset and seed. For each damaged training set the meta-features are read from the Notebook 7 file (after checking that the number of rows and the missing rate match); otherwise they are computed again. The three larger models are trained on the clean training set (time recorded) and on every damaged copy.

In [ ]:
def n7_path(did, seed):
    return os.path.join(N7_DIR, 'n7_' + str(did) + '_seed' + str(seed) + '.csv')


def new_path(did, seed):
    return os.path.join(OUTPUT_DIR, FILE_PREFIX + '_' + str(did) + '_seed' + str(seed) + '.csv')


def run_dataset_seed(did, seed, X, y, name, round1):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
    X_tr = X_tr.reset_index(drop=True)
    clean, tfit = {}, {}
    for m in ALL_MODELS:
        t0 = time.perf_counter()
        clean[m] = scores(y_te, fit_predict_big(m, X_tr, y_tr, X_te))
        tfit[m] = time.perf_counter() - t0
    ref = None
    if os.path.exists(n7_path(did, seed)):
        ref = pd.read_csv(n7_path(did, seed))
        ref = ref[ref['model'] == 'rf']
    rows = []
    n_reused = 0
    for sid, setting in enumerate(SETTINGS):
        if not keep_setting(sid, round1):
            continue
        Xd, yd = apply_defects(X_tr, y_tr, setting, seed)
        mf = None
        if ref is not None:
            o = ref[ref['setting_id'] == sid]
            if len(o) == 1 and abs(o['n_rows'].iloc[0] - len(yd)) < 0.5 and abs(o['missing_rate'].iloc[0] - float(Xd.isna().to_numpy().mean())) < 1e-6:
                mf = {c: o[c].iloc[0] for c in META_COLS + CL_COLS + ['t_meta', 't_landmark', 't_cl'] if c in o.columns}
                n_reused += 1
        if mf is None:
            mf = all_meta_features(Xd, yd, seed)
        for m in ALL_MODELS:
            sc = clean[m] if not setting else scores(y_te, fit_predict_big(m, Xd, yd, X_te))
            row = {'dataset': name, 'dataset_id': did, 'seed': seed, 'setting_id': sid, 'model': m, 't_fit': tfit[m]}
            for d in DEFECTS:
                row['inj_' + d] = setting.get(d, 0.0)
            row.update(mf)
            for k in METRICS:
                row['deg_' + k] = clean[m][k] - sc[k]
            rows.append(row)
    return pd.DataFrame(rows), n_reused


failed = []
for did in DATASET_IDS:
    todo = [s for s in SEEDS if not os.path.exists(new_path(did, s))]
    if not todo:
        continue
    try:
        X, y, name = load_dataset(did)
    except Exception as e:
        print('FAILED to load dataset', did, ':', repr(e)[:200])
        failed.append(did)
        continue
    for seed in todo:
        t0 = time.time()
        try:
            df, nre = run_dataset_seed(did, seed, X, y, name, did in ORIGINAL_IDS)
        except Exception as e:
            print('FAILED dataset', did, 'seed', seed, ':', repr(e)[:200])
            failed.append(did)
            break
        df.to_csv(new_path(did, seed), index=False)
        print(f'done {did} ({name}), seed {seed}, in {(time.time() - t0) / 60:.1f} min; meta-features re-used for {nre} settings')
print('Failed datasets:', sorted(set(failed)))

## Part C: prepare the results

A dataset is used if all three seeds finished. Seeds are averaged. The standard-size models were trained in the run above on the **same data, seeds and settings**, so the two groups are compared fairly.

In [ ]:
files = sorted(glob.glob(os.path.join(OUTPUT_DIR, FILE_PREFIX + '_*_seed*.csv')))
big = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
big = big[big['dataset_id'].isin(DATASET_IDS) & big['seed'].isin(SEEDS)]
have = big.groupby('dataset_id')['seed'].apply(lambda s: set(s.unique()))
good_ids = [d for d, s in have.items() if set(SEEDS) <= s]
big = big[big['dataset_id'].isin(good_ids)]
ORIG = [d for d in ORIGINAL_IDS if d in good_ids]
print('Datasets used:', len(good_ids), '| Round 1:', len(ORIG), '| Round 2:', len(good_ids) - len(ORIG))

std = big[big['model'].isin(STD_MODELS)]
STD_OK = len(std) > 0

ALLF = META_COLS + CL_COLS
key = ['dataset', 'dataset_id', 'setting_id']
meta = (big[big['model'] == 'rf_big'].groupby(key, as_index=False)[ALLF + ['inj_' + d for d in DEFECTS]].mean())


def defect_type(row):
    active = [d for d in DEFECTS if row['inj_' + d] > 0]
    return 'clean' if not active else (active[0] if len(active) == 1 else 'combined')


meta['defect_type'] = meta.apply(defect_type, axis=1)
meta['level'] = [r['inj_' + r['defect_type']] if r['defect_type'] not in ('combined', 'clean') else r['inj_missing']
                 for _, r in meta.iterrows()]


def model_frame(src_df, model):
    d = src_df[src_df['model'] == model].groupby(key, as_index=False)[['deg_' + k for k in METRICS]].mean()
    return meta.merge(d, on=key)


FRAMES = {m: model_frame(big, m) for m in BIG_MODELS}
if STD_OK:
    for m in STD_MODELS:
        FRAMES[m] = model_frame(std, m)
for m in FRAMES:
    FRAMES[m] = FRAMES[m][FRAMES[m]['setting_id'] != 0].reset_index(drop=True)
print({m: len(f) for m, f in FRAMES.items()})

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr, wilcoxon

SMALL = ['missing_rate', 'duplicate_rate', 'imbalance_ratio', 'class_entropy', 'lm_stump', 'lm_tree3', 'lm_nb']
LM = ['lm_stump', 'lm_tree3', 'lm_nb']


def per_ds(y_true, y_pred, ids):
    return pd.Series(np.abs(np.asarray(y_true) - np.asarray(y_pred))).groupby(np.asarray(ids)).mean()


def wil(a, b, alt='less'):
    try:
        return wilcoxon(a, b, alternative=alt).pvalue
    except Exception:
        return np.nan


def holm(pvals):
    # Holm step-down correction; returns adjusted p-values in the original order
    p = np.asarray(pvals, dtype=float)
    out = np.full(len(p), np.nan)
    ok = ~np.isnan(p)
    idx = np.where(ok)[0][np.argsort(p[ok])]
    m = len(idx)
    run = 0.0
    for rank, i in enumerate(idx):
        run = max(run, (m - rank) * p[i])
        out[i] = min(1.0, run)
    return out


def boot_ci(err_meta, err_base, n=2000, seed=0):
    # 95% bootstrap interval (resampling datasets) of the relative error reduction of the meta-model
    rng = np.random.default_rng(seed)
    a, b = np.asarray(err_meta), np.asarray(err_base)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(a), len(a))
        vals.append(1 - a[i].mean() / b[i].mean())
    return np.percentile(vals, [2.5, 97.5])
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.cluster import KMeans

SMALL_CL = SMALL + CL_COLS
LM_CL = LM + CL_COLS
METHODS = {
    'rf_small': ('rf', SMALL),
    'rf_small_cl': ('rf', SMALL_CL),
    'rf_lm_cl': ('rf', LM_CL),
    'rf_all': ('rf', ALLF),
    'ridge_all': ('ridge', ALLF),
    'gb_all': ('gb', ALLF),
    'cl_only': ('iso', ['cl_noise_rate']),
    'cl_issues_only': ('iso', ['cl_issue_rate']),
    'rf_small_told': ('rf', SMALL + ['inj_label_noise']),
}


def per_ds(y_true, y_pred, ids):
    return pd.Series(np.abs(np.asarray(y_true) - np.asarray(y_pred))).groupby(np.asarray(ids)).mean()


def wil(a, b, alt='less'):
    try:
        return wilcoxon(a, b, alternative=alt).pvalue
    except Exception:
        return np.nan


def holm(pvals):
    # Holm step-down correction; returns adjusted p-values in the original order
    p = np.asarray(pvals, dtype=float)
    out = np.full(len(p), np.nan)
    ok = ~np.isnan(p)
    idx = np.where(ok)[0][np.argsort(p[ok])]
    m = len(idx)
    run = 0.0
    for rank, i in enumerate(idx):
        run = max(run, (m - rank) * p[i])
        out[i] = min(1.0, run)
    return out


def boot_ci(err_meta, err_base, n=2000, seed=0):
    # 95% bootstrap interval (resampling datasets) of the relative error reduction of the meta-model
    rng = np.random.default_rng(seed)
    a, b = np.asarray(err_meta), np.asarray(err_base)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(a), len(a))
        vals.append(1 - a[i].mean() / b[i].mean())
    return np.percentile(vals, [2.5, 97.5])
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.cluster import KMeans

SMALL_CL = SMALL + CL_COLS
LM_CL = LM + CL_COLS
METHODS = {
    'rf_small': ('rf', SMALL),
    'rf_small_cl': ('rf', SMALL_CL),
    'rf_lm_cl': ('rf', LM_CL),
    'rf_all': ('rf', ALLF),
    'ridge_all': ('ridge', ALLF),
    'gb_all': ('gb', ALLF),
    'cl_only': ('iso', ['cl_noise_rate']),
    'cl_issues_only': ('iso', ['cl_issue_rate']),
    'rf_small_told': ('rf', SMALL + ['inj_label_noise']),
}


def make_meta(kind='rf'):
    imp = SimpleImputer(strategy='median', keep_empty_features=True)
    if kind == 'ridge':
        return make_pipeline(imp, StandardScaler(), Ridge(alpha=1.0))
    if kind == 'gb':
        return make_pipeline(imp, GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.05,
                                                            subsample=0.8, random_state=0))
    return make_pipeline(imp, RandomForestRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.5,
                                                    n_jobs=-1, random_state=0))


def predict_method(name, tr, te, y_tr):
    kind, cols = METHODS[name]
    if kind == 'iso':
        x_tr = tr[cols[0]].to_numpy(dtype=float)
        x_te = te[cols[0]].to_numpy(dtype=float)
        fill = np.nanmedian(x_tr) if np.isfinite(x_tr).any() else 0.0
        x_tr = np.where(np.isnan(x_tr), fill, x_tr)
        x_te = np.where(np.isnan(x_te), fill, x_te)
        return IsotonicRegression(out_of_bounds='clip').fit(x_tr, y_tr).predict(x_te)
    return make_meta(kind).fit(tr[cols], y_tr).predict(te[cols])


def told_noise_pred(tr, te, k):
    trl = tr[tr['defect_type'] == 'label_noise']
    tab = trl.groupby('level')['deg_' + k].mean()
    return np.array([tab.iloc[np.abs(tab.index.to_numpy() - l).argmin()] for l in te['level']])


def planned_split(m):
    tr = data[(data['model'] == m) & data['dataset_id'].isin(ORIG) & (data['defect_type'] != 'class_noise')]
    te = data[(data['model'] == m) & ~data['dataset_id'].isin(ORIG) & (data['defect_type'] == 'label_noise')].reset_index(drop=True)
    return tr, te


def summarize(y_te, ids, preds, base='avg_drop'):
    # per-method error, error reduction against the baseline with a bootstrap interval, wins and p-value
    e = {n: per_ds(y_te, p, ids) for n, p in preds.items()}
    out = []
    for n, p in preds.items():
        row = {'method': n, 'MAE': float(np.abs(y_te - p).mean())}
        if n != base:
            lo, hi = boot_ci(e[n], e[base])
            row.update({'reduction_%': 100 * (1 - e[n].mean() / e[base].mean()), 'CI95_low_%': 100 * lo, 'CI95_high_%': 100 * hi,
                        'better_on': str(int((e[n] < e[base]).sum())) + '/' + str(len(e[n])),
                        'p_one_sided': wil(e[n], e[base]), 'rank_corr': spearmanr(y_te, p)[0]})
        out.append(row)
    return out
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression

SMALL_CL = SMALL + CL_COLS


def make_meta():
    return make_pipeline(SimpleImputer(strategy='median', keep_empty_features=True),
                         RandomForestRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.5,
                                               n_jobs=-1, random_state=0))


def fit_meta(cols, tr, y_tr, te):
    return make_meta().fit(tr[cols], y_tr).predict(te[cols])


def told_noise_pred(tr, te, k):
    trl = tr[tr['defect_type'] == 'label_noise']
    tab = trl.groupby('level')['deg_' + k].mean()
    return np.array([tab.iloc[np.abs(tab.index.to_numpy() - l).argmin()] for l in te['level']])


def planned_split(frame):
    tr = frame[frame['dataset_id'].isin(ORIG) & (frame['defect_type'] != 'class_noise')]
    te = frame[~frame['dataset_id'].isin(ORIG) & (frame['defect_type'] == 'label_noise')].reset_index(drop=True)
    return tr, te


def summarize(y_te, ids, preds, base='avg_drop'):
    e = {n: per_ds(y_te, p, ids) for n, p in preds.items()}
    out = []
    for n, p in preds.items():
        row = {'method': n, 'MAE': float(np.abs(y_te - p).mean())}
        if n != base:
            lo, hi = boot_ci(e[n], e[base])
            row.update({'reduction_%': 100 * (1 - e[n].mean() / e[base].mean()), 'CI95_low_%': 100 * lo, 'CI95_high_%': 100 * hi,
                        'better_on': str(int((e[n] < e[base]).sum())) + '/' + str(len(e[n])),
                        'p_one_sided': wil(e[n], e[base]), 'rank_corr': spearmanr(y_te, p)[0]})
        out.append(row)
    return out

## Results 1: the planned test for standard and larger models

Same test as in the paper: train the meta-model on the Round-1 datasets, test on label noise in the Round-2 datasets. `reduction_%` is the drop in error against the average-drop baseline (positive = better), with a 95% bootstrap interval over datasets. `rf_small` is the paper's meta-model; `rf_small_cl` adds the four cleanlab features; `told_noise` is the baseline that is told the true noise level. Holm correction is over all rows of the table.

In [ ]:
rows = []
for k in ['acc', 'f1']:
    for m in FRAMES:
        tr, te = planned_split(FRAMES[m])
        if len(te) == 0 or len(tr) == 0:
            continue
        y_tr, y_te, ids = tr['deg_' + k].to_numpy(), te['deg_' + k].to_numpy(), te['dataset_id'].to_numpy()
        preds = {'avg_drop': np.full(len(te), y_tr.mean()), 'told_noise': told_noise_pred(tr, te, k),
                 'rf_small': fit_meta(SMALL, tr, y_tr, te), 'rf_small_cl': fit_meta(SMALL_CL, tr, y_tr, te)}
        for r in summarize(y_te, ids, preds):
            if r['method'] != 'avg_drop':
                r.update({'measure': METRIC_NAMES[k], 'model': m, 'size': 'larger' if m in BIG_MODELS else 'standard',
                          'MAE_avg_drop': float(np.abs(y_te - preds['avg_drop']).mean())})
                rows.append(r)
res1 = pd.DataFrame(rows)
res1['p_holm'] = holm(res1['p_one_sided'].to_numpy())
res1 = res1[['measure', 'model', 'size', 'method', 'MAE_avg_drop', 'MAE', 'reduction_%', 'CI95_low_%', 'CI95_high_%',
             'better_on', 'p_one_sided', 'p_holm', 'rank_corr']].round(4)
res1.to_csv(os.path.join(OUTPUT_DIR, 'result8_1_planned_test_larger_models.csv'), index=False)
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)
res1

## Results 2: do larger models lose more or less accuracy from label noise?

Average loss over datasets, at each label-noise level, for the standard and the larger model of each kind (accuracy). `paired_p` is a two-sided Wilcoxon test over datasets (larger versus standard loss at the same level). This shows whether the quantity the meta-model predicts is itself different for larger models.

In [ ]:
rows = []
for big_m, std_m in PAIR.items():
    if std_m not in FRAMES:
        continue
    for lv in sorted(FRAMES[big_m].loc[FRAMES[big_m]['defect_type'] == 'label_noise', 'level'].unique()):
        a = FRAMES[std_m][(FRAMES[std_m]['defect_type'] == 'label_noise') & (FRAMES[std_m]['level'] == lv)].set_index('dataset_id')['deg_acc']
        b = FRAMES[big_m][(FRAMES[big_m]['defect_type'] == 'label_noise') & (FRAMES[big_m]['level'] == lv)].set_index('dataset_id')['deg_acc']
        both = a.index.intersection(b.index)
        try:
            p = wilcoxon(b[both], a[both], alternative='two-sided').pvalue
        except Exception:
            p = np.nan
        rows.append({'kind': std_m, 'noise_level': lv, 'datasets': len(both), 'mean_loss_standard': a[both].mean(),
                     'mean_loss_larger': b[both].mean(), 'larger_loses_more_on': int((b[both] > a[both]).sum()), 'paired_p': p})
res2 = pd.DataFrame(rows).round(4)
res2.to_csv(os.path.join(OUTPUT_DIR, 'result8_2_loss_standard_vs_larger.csv'), index=False)
res2

## Results 3: can a meta-model trained on standard-size models predict the loss of larger models?

The meta-model is trained with the **standard** models' loss as the target on Round 1, then asked to predict the loss of the **larger** model on the Round-2 label-noise cases. If this works, the meta-model does not need to be retrained for every model size. The baseline is the average standard-model loss in training.

In [ ]:
rows = []
for k in ['acc', 'f1']:
    for big_m, std_m in PAIR.items():
        if std_m not in FRAMES:
            continue
        tr_s, _ = planned_split(FRAMES[std_m])
        _, te_b = planned_split(FRAMES[big_m])
        if len(te_b) == 0 or len(tr_s) == 0:
            continue
        y_tr = tr_s['deg_' + k].to_numpy()
        y_te, ids = te_b['deg_' + k].to_numpy(), te_b['dataset_id'].to_numpy()
        preds = {'avg_drop': np.full(len(te_b), y_tr.mean()), 'told_noise': told_noise_pred(tr_s, te_b, k),
                 'rf_small': fit_meta(SMALL, tr_s, y_tr, te_b), 'rf_small_cl': fit_meta(SMALL_CL, tr_s, y_tr, te_b)}
        for r in summarize(y_te, ids, preds):
            if r['method'] != 'avg_drop':
                r.update({'measure': METRIC_NAMES[k], 'trained_on': std_m, 'tested_on': big_m})
                rows.append(r)
res3 = pd.DataFrame(rows)
if len(res3) == 0:
    print('Skipped: needs the standard-size models (Notebook 7 files).')
    res3 = pd.DataFrame(columns=['measure', 'trained_on', 'tested_on', 'method', 'p_one_sided'])
res3['p_holm'] = holm(res3['p_one_sided'].to_numpy())
res3 = res3[[c for c in ['measure', 'trained_on', 'tested_on', 'method', 'MAE', 'reduction_%', 'CI95_low_%', 'CI95_high_%',
             'better_on', 'p_one_sided', 'p_holm', 'rank_corr'] if c in res3.columns]].round(4)
res3.to_csv(os.path.join(OUTPUT_DIR, 'result8_3_transfer_standard_to_larger.csv'), index=False)
res3

## Results 4: training time of the larger models against the meta-features

Median seconds per dataset and seed. Training time is one fit of the model on the clean training set. `meta_features_s` is the time of all 20 meta-features (16 plus the 4 cleanlab ones), `landmark_s` that of the three landmarking learners (the costly part of the 7-feature meta-model). `share_faster` is the share of dataset-seed pairs where the features are quicker than training that model. This uses the clean-data rows of the larger models.

In [ ]:
cols = ['t_meta', 't_landmark', 't_cl']
if 't_meta' in big.columns and big['t_meta'].notna().any():
    tm = big[big['setting_id'] > 0].groupby(['dataset_id', 'seed'], as_index=False)[cols].mean()
    tm['t_features'] = tm['t_meta'] + tm['t_cl']
    fit = big.groupby(['dataset_id', 'seed', 'model'], as_index=False)['t_fit'].mean()
    tt = fit.merge(tm, on=['dataset_id', 'seed'])
    rows = []
    for m in BIG_MODELS:
        d = tt[tt['model'] == m]
        rows.append({'model': m, 'training_s': d['t_fit'].median(), 'meta_features_s': d['t_features'].median(),
                     'landmark_s': d['t_landmark'].median(),
                     'share_faster_all_features': float((d['t_features'] < d['t_fit']).mean()),
                     'share_faster_landmark_only': float((d['t_landmark'] < d['t_fit']).mean()),
                     'median_ratio_all': float((d['t_features'] / d['t_fit']).median()),
                     'median_ratio_landmark': float((d['t_landmark'] / d['t_fit']).median())})
    res4 = pd.DataFrame(rows).round(3)
    res4.to_csv(os.path.join(OUTPUT_DIR, 'result8_4_timing_larger_models.csv'), index=False)
else:
    res4 = pd.DataFrame()
    print('No timing columns found.')
res4

## Done

Send back the four files `result8_1_...csv` to `result8_4_...csv` from the results folder (and the notebook output if you can). Then the paper gets one short paragraph and a table on larger models.